# Lesson 02 — Computing Homography from Feature Matches

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt

img1 = cv2.imread('sample.jpg')
img2 = img1.copy()
M_sim = cv2.getRotationMatrix2D((img1.shape[1]//2,img1.shape[0]//2),20,0.85)
img2  = cv2.warpAffine(img2, M_sim, (img2.shape[1],img2.shape[0]))

sift = cv2.SIFT_create(nfeatures=500)
kp1,d1 = sift.detectAndCompute(cv2.cvtColor(img1,cv2.COLOR_BGR2GRAY),None)
kp2,d2 = sift.detectAndCompute(cv2.cvtColor(img2,cv2.COLOR_BGR2GRAY),None)
bf     = cv2.BFMatcher()
good   = [m for m,n in bf.knnMatch(d1,d2,k=2) if m.distance < 0.75*n.distance]

if len(good) >= 4:
    src = np.float32([kp1[m.queryIdx].pt for m in good]).reshape(-1,1,2)
    dst = np.float32([kp2[m.trainIdx].pt for m in good]).reshape(-1,1,2)
    H, mask = cv2.findHomography(src, dst, cv2.RANSAC, 5.0)
    inliers = mask.ravel().sum()
    print(f"Homography from {len(good)} matches: {inliers} inliers, {len(good)-inliers} outliers")
    print(f"H matrix:\n{H}")

    aligned = cv2.warpPerspective(img1, H, (img2.shape[1], img2.shape[0]))
    blend   = cv2.addWeighted(img2, 0.5, aligned, 0.5, 0)

    plt.figure(figsize=(16,5))
    plt.subplot(1,3,1); plt.imshow(cv2.cvtColor(img1,    cv2.COLOR_BGR2RGB)); plt.title('Image 1'); plt.axis('off')
    plt.subplot(1,3,2); plt.imshow(cv2.cvtColor(img2,    cv2.COLOR_BGR2RGB)); plt.title('Image 2'); plt.axis('off')
    plt.subplot(1,3,3); plt.imshow(cv2.cvtColor(blend,   cv2.COLOR_BGR2RGB)); plt.title('Aligned (50% blend)'); plt.axis('off')
    plt.show()

## Key Takeaway
`findHomography(src, dst, cv2.RANSAC, reprojThresh)` — RANSAC flag rejects outlier matches.
The returned `mask` tells you which matches were inliers. inlier_ratio > 0.5 = reliable homography.